# 05 QLoRA Low Resource

## Overview

This notebook explains what QLoRA adds on top of LoRA.
You will configure 4 bit loading, estimate memory savings, and run the same small fine tuning flow when CUDA is available.

Estimated time: 30 to 50 minutes on a CUDA GPU.

## Prerequisites

Complete notebook 04 first.
You should already understand the basic LoRA flow.

In [ ]:
%pip install -q transformers==4.40.0 peft==0.10.0 trl==0.8.6 bitsandbytes==0.43.1 datasets==2.19.0 torch==2.2.2 accelerate==0.29.3

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## Why QLoRA Matters

QLoRA keeps the adapter idea from LoRA, then compresses the frozen base model weights to 4 bit values.
That reduces VRAM use enough to make larger models practical on smaller GPUs.

In [ ]:
model_params = 1.1e9
bytes_fp16 = model_params * 2
bytes_4bit = model_params * 0.5

print(f"Estimated fp16 weight memory: {bytes_fp16 / 1024**3:.2f} GB")
print(f"Estimated 4 bit weight memory: {bytes_4bit / 1024**3:.2f} GB")

In [ ]:
from transformers import BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

quant_config

In [ ]:
if device != "cuda":
    print("QLoRA training is skipped on CPU because bitsandbytes 4 bit training requires CUDA.")
else:
    from pathlib import Path
    import sys
    import json

    sys.path.append(str(Path("..").resolve()))

    from datasets import Dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from trl import SFTTrainer
    from utils.helpers import DATASETS_DIR, QLORA_ADAPTER_DIR, QLORA_CHECKPOINTS_DIR, format_alpaca

    model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quant_config,
        device_map="auto",
    )

    model = prepare_model_for_kbit_training(model)

    config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "v_proj"],
    )
    model = get_peft_model(model, config)

    rows = [
        json.loads(line)
        for line in (DATASETS_DIR / "sample_dataset.jsonl").read_text(encoding="utf-8").splitlines()[:8]
        if line.strip()
    ]
    dataset = Dataset.from_list([
        {"text": format_alpaca(item["instruction"], item["input"], item["output"])}
        for item in rows
    ])

    args = TrainingArguments(
        output_dir=str(QLORA_CHECKPOINTS_DIR),
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=1,
        save_strategy="no",
        fp16=True,
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        dataset_text_field="text",
        max_seq_length=256,
        args=args,
    )
    trainer.train()
    trainer.model.save_pretrained(QLORA_ADAPTER_DIR)
    tokenizer.save_pretrained(QLORA_ADAPTER_DIR)
    print(f"Saved QLoRA adapter to {QLORA_ADAPTER_DIR}")

## Key Takeaways

* QLoRA combines LoRA adapters with 4 bit loading.
* The big win is lower VRAM use, not magic quality gains.
* QLoRA needs CUDA in this workflow.
* Once you understand notebook 04, QLoRA is a practical upgrade path for bigger models.

## Next Steps

Continue to [06 Evaluating Your Model](../06-evaluating-your-model/06-evaluating-your-model.ipynb).